In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

## Dataset Paths

In [ ]:
BASE_DIR       = "/kaggle/input/competitions/recodai-luc-scientific-image-forgery-detection"

TRAIN_IMG_DIR  = os.path.join(BASE_DIR, "train_images")
TRAIN_MASK_DIR = os.path.join(BASE_DIR, "train_masks")
SUPP_IMG_DIR   = os.path.join(BASE_DIR, "supplemental_images")
SUPP_MASK_DIR  = os.path.join(BASE_DIR, "supplemental_masks")
TEST_IMG_DIR   = os.path.join(BASE_DIR, "test_images")

# Verify all directories exist
for p in [TRAIN_IMG_DIR, TRAIN_MASK_DIR, SUPP_IMG_DIR, SUPP_MASK_DIR, TEST_IMG_DIR]:
    print(p, "→", os.path.exists(p))

In [ ]:
# Output paths for saved model and submission
best_model_path = "/kaggle/working/best_model.pth"
submission_path = "/kaggle/working/submission.csv"

## Load & Match Image-Mask Pairs

In [ ]:
# Only forged images have corresponding masks
FORGED_DIR = os.path.join(TRAIN_IMG_DIR, "forged")

train_images = sorted(glob(f"{FORGED_DIR}/*.png"))
train_masks  = sorted(glob(f"{TRAIN_MASK_DIR}/*.npy"))

# Match images and masks by filename stem
image_dict = {os.path.splitext(os.path.basename(x))[0]: x for x in train_images}
mask_dict  = {os.path.splitext(os.path.basename(x))[0]: x for x in train_masks}

common_ids   = sorted(set(image_dict.keys()) & set(mask_dict.keys()))
train_images = [image_dict[i] for i in common_ids]
train_masks  = [mask_dict[i]  for i in common_ids]

print("Matched image-mask pairs:", len(train_images))

In [ ]:
# Inspect a sample mask to confirm shape and dtype
mask = np.load(train_masks[0])
print("Mask shape:", mask.shape)
print("Mask dtype:", mask.dtype)

## Metrics evaluation

In [ ]:
def compute_metrics(preds_binary, targets_binary):
    """Compute IoU, Dice/F1, Precision, Recall, and Pixel Accuracy."""
    preds   = preds_binary.bool()
    targets = targets_binary.bool()

    TP = (preds & targets).sum().item()
    FP = (preds & ~targets).sum().item()
    FN = (~preds & targets).sum().item()
    TN = (~preds & ~targets).sum().item()
    total = TP + FP + FN + TN

    return {
        "IoU":       round(TP / (TP + FP + FN + 1e-8), 4),
        "Dice/F1":   round((2 * TP) / (2 * TP + FP + FN + 1e-8), 4),
        "Precision": round(TP / (TP + FP + 1e-8), 4),
        "Recall":    round(TP / (TP + FN + 1e-8), 4),
        "Pixel Acc": round((TP + TN) / (total + 1e-8), 4),
    }

## Dataset & Transforms

In [ ]:
class ForgeryDataset(Dataset):
    """Loads forged images and their binary segmentation masks."""

    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image in RGB
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load .npy mask (first channel)
        mask = np.load(self.mask_paths[idx])[0]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]

        # Add channel dim to mask: (H,W) -> (1,H,W)
        mask = mask.float().unsqueeze(0)
        return image, mask

In [ ]:
# Training augmentations; validation only resizes and normalizes
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])

## Train / Validation Split

In [ ]:
train_imgs, val_imgs, train_msks, val_msks = train_test_split(
    train_images, train_masks,
    test_size=0.2, random_state=42
)

print("Train:",      len(train_imgs))
print("Validation:", len(val_imgs))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

train_dataset = ForgeryDataset(train_imgs, train_msks, transform=train_transform)
val_dataset   = ForgeryDataset(val_imgs,   val_msks,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=0)

# Confirm batch shapes
images, masks = next(iter(train_loader))
print("Image batch:", images.shape)
print("Mask batch: ", masks.shape)

## Model — UNet + EfficientNet-B0

In [ ]:
model = smp.Unet(
    encoder_name="efficientnet-b0",
    encoder_weights="imagenet",  # pretrained on ImageNet
    in_channels=3,
    classes=1                    # binary segmentation
).to(DEVICE)

In [ ]:
loss_fn   = smp.losses.DiceLoss(mode="binary")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## Training

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

# Save model after training
torch.save(model.state_dict(), best_model_path)
print("Model saved to", best_model_path)

## Validation Metrics

In [ ]:
model.eval()
all_preds, all_masks_list = [], []

with torch.no_grad():
    for images, masks in val_loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        preds   = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.append(preds.cpu())
        all_masks_list.append(masks.cpu())

all_preds      = torch.cat(all_preds)
all_masks_eval = torch.cat(all_masks_list)

metrics = compute_metrics(all_preds, all_masks_eval)
print(metrics)

## Visualisation — Sample Predictions

In [ ]:
model.eval()

sample_imgs, sample_masks = next(iter(val_loader))
sample_imgs = sample_imgs.to(DEVICE)

with torch.no_grad():
    logits = model(sample_imgs)
    probs  = torch.sigmoid(logits)
    preds  = (probs > 0.5).float()

# Undo ImageNet normalisation for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for idx in range(4):
    img       = (sample_imgs[idx].cpu() * std + mean).permute(1, 2, 0).numpy().clip(0, 1)
    gt_mask   = sample_masks[idx, 0].numpy()
    pred_mask = preds[idx, 0].cpu().numpy()

    axes[idx][0].imshow(img);       axes[idx][0].set_title("Input Image");    axes[idx][0].axis("off")
    axes[idx][1].imshow(gt_mask,   cmap="gray"); axes[idx][1].set_title("Ground Truth");  axes[idx][1].axis("off")
    axes[idx][2].imshow(pred_mask, cmap="gray"); axes[idx][2].set_title("Prediction");    axes[idx][2].axis("off")

plt.tight_layout()
plt.show()

## Generating for Submission

In [ ]:
def rle_encode(mask):
    """Run length Encode a binary mask for submission."""
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

In [ ]:
model.eval()
predictions = []

test_images = sorted(glob(f"{TEST_IMG_DIR}/*.png"))
print("Total test images:", len(test_images))

for i, img_path in enumerate(test_images):
    image     = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    input_tensor = val_transform(image=image_rgb)["image"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output    = model(input_tensor)
        pred      = (torch.sigmoid(output) > 0.5).float()
        pred_mask = pred.squeeze().cpu().numpy().astype(np.uint8)

    rle      = rle_encode(pred_mask)
    image_id = os.path.basename(img_path)
    predictions.append([image_id, rle])

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(test_images)} done")

print("Predictions generated:", len(predictions))

In [ ]:
submission_df = pd.DataFrame(predictions, columns=["id", "rle"])
submission_df.to_csv(submission_path, index=False)

print(submission_df.head())
print(f"Saved to {submission_path}")